# KORE 데이터 작업 (B·C용, GPU 필요 없음) — 코드 zip 판

원래 노트북 03은 GitHub 저장소에서 코드를 받아 온다. 이 판은 GitHub 없이 **KORE_코드_8f70421.zip을 직접 올려서** 같은 일을 한다. 아래 3-B, 3-C, 4번 셀은 원래 노트북과 같다.

순서: 1 → 2 → (B는 3-B, C는 3-C) → 4.
- 1번 셀: 코드 zip을 올리면 풀고 설치한다.
- 2번 셀: B는 원문 txt, C는 `evaluation_raw.csv`와 **B가 보낸 `knowledge.csv`, `source_log.csv`** 를 올린다(질문의 근거 문서 id를 확인하려면 문서 파일이 필요하다).
- 3번 셀: 변환 또는 분할과 검증을 한다.
- 4번 셀: 결과 CSV를 내려받아 서준에게 보낸다. Colab에서 GitHub로 직접 올리지 않는다.

주의: Colab 런타임은 닫으면 파일이 사라진다. 여러 번에 나눠 작업할 때는 지난번에 내려받은 `knowledge.csv`, `source_log.csv`를 2번 셀로 먼저 다시 올리고 시작한다.

## 1. 준비

In [ ]:
# 1. 준비: 코드 zip 올리기, 풀기, 설치 (GitHub 없이)
import os, sys, glob, shutil, subprocess, zipfile

WORKDIR = '/content/kore'

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print((r.stdout or '')[-3000:] + (r.stderr or '')[-1500:])
    if r.returncode != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (r.returncode, cmd))

def setup_from_zip(zip_path, workdir=WORKDIR):
    """zip을 풀어 pyproject.toml이 있는 폴더를 workdir로 옮긴다."""
    tmp = workdir + '_unzip'
    for d in (tmp, workdir):
        if os.path.isdir(d):
            shutil.rmtree(d)
    os.makedirs(tmp)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(tmp)
    hits = glob.glob(os.path.join(tmp, '**', 'pyproject.toml'), recursive=True)
    if not hits:
        raise SystemExit('zip 안에서 pyproject.toml을 찾지 못했다. KORE_코드_8f70421.zip이 맞는지 확인한다.')
    root = min((os.path.dirname(h) for h in hits), key=len)
    shutil.move(root, workdir)
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(os.path.join(workdir, 'raw'), exist_ok=True)
    os.makedirs(os.path.join(workdir, 'data'), exist_ok=True)

if 'google.colab' in sys.modules or os.environ.get('COLAB_RELEASE_TAG'):
    from google.colab import files
    print('KORE_코드_8f70421.zip 파일을 선택해서 올린다.')
    up = files.upload()
    zips = [n for n in up if n.lower().endswith('.zip')]
    if not zips:
        raise SystemExit('zip 파일이 올라오지 않았다. 이 셀을 다시 실행해서 zip을 고른다.')
    setup_from_zip(zips[0])
    sh(sys.executable + ' -m pip install -q -e .', cwd=WORKDIR)
    sh(sys.executable + ' -m kore doctor', cwd=WORKDIR)
    print('준비 끝. GPU는 필요 없다(런타임 유형: CPU).')


## 2. 파일 올리기
B: 원문 txt (이전에 만든 knowledge.csv, source_log.csv가 있으면 같이)
C: evaluation_raw.csv, 그리고 B가 보낸 knowledge.csv, source_log.csv


In [ ]:
# 2. 내 컴퓨터의 파일을 Colab으로 올린다
# B: 원문 txt → raw/ 폴더,  C: evaluation_raw.csv → data/ 폴더
import os, shutil
from google.colab import files
WORKDIR = '/content/kore'
up = files.upload()
for name in up:
    dest = 'data' if name.endswith('.csv') else 'raw'
    shutil.move(name, os.path.join(WORKDIR, dest, name))
    print('올림:', dest + '/' + name)

## 3-B. 문서 올리기 (B)
`DRY_RUN = True`로 먼저 보고, 잘린 모양이 맞으면 `False`로 바꿔 다시 실행한다.

In [ ]:
# 3-B. 원문을 문서 CSV로
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

os.chdir(WORKDIR)
# ---- 여기만 바꾼다 ----
INPUT = 'raw/law_labor.txt'               # 2번 셀로 올린 원문
DOMAIN = 'law'                            # law / guide / wiki
ID_PREFIX = 'law_labor'                   # 문서 id 앞부분(파일마다 다르게)
SOURCE_URL = 'https://www.law.go.kr/...'  # 원문 주소
LICENSE = '저작권법 제7조(법령)'            # 직접 확인한 이용 조건. 모르면 unknown
MODE = 'article'                          # 법령은 article, 제목이 있는 안내문은 heading, 그 밖은 paragraph
ACCESSED = '2026-10-24'                   # 원문을 연 날짜
DRY_RUN = True                            # 먼저 True로 잘리는 모양을 보고, 괜찮으면 False로 바꿔 다시 실행
# ----------------------
cmd = (sys.executable + ' -m kore ingest --input "%s" --domain %s --id-prefix %s --source-url "%s" --license "%s" '
       '--mode %s --out data/knowledge.csv --append --accessed-date %s') % (INPUT, DOMAIN, ID_PREFIX, SOURCE_URL, LICENSE, MODE, ACCESSED)
sh(cmd + (' --dry-run' if DRY_RUN else ''), stream=True)
if not DRY_RUN:
    sh(sys.executable + ' -m kore validate --knowledge data/knowledge.csv', stream=True)
    sh(sys.executable + ' -m kore stats --knowledge data/knowledge.csv --stage week1', stream=True)
    print('다음: 4번 셀로 knowledge.csv와 source_log.csv를 내려받아 GitHub에 올린다(START.md 6절).')

## 3-C. 질문 나누기와 점검 (C)
`evaluation_raw.csv`를 2번 셀로 올린 뒤 실행한다.

In [ ]:
# 3-C. 분할과 점검
import os, subprocess, sys, time

REPO_URL = 'https://github.com/OWNER/kore.git'  # 팀 저장소 주소로 바꾼다(Secrets에 KORE_REPO_URL을 저장했으면 그대로 둬도 된다)
try:
    from google.colab import userdata
    REPO_URL = userdata.get('KORE_REPO_URL') or REPO_URL
except Exception:
    pass
BRANCH = 'main'
WORKDIR = '/content/kore'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')  # Colab 왼쪽 Secrets에 저장한 읽기 전용 토큰(비공개 저장소일 때만)
    except Exception:
        return None

def sh(cmd, cwd=None, stream=False, secret=None):
    """명령을 실행하고 출력을 보여 준다. secret이 출력에 섞이면 가린다."""
    mask = (lambda s: s.replace(secret, '***')) if secret else (lambda s: s)
    if stream:
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(mask(line), end='')
        p.wait()
        rc = p.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print(mask(r.stdout[-3000:] + r.stderr[-1500:]))
        rc = r.returncode
    if rc != 0:
        raise RuntimeError('명령 실패(코드 %s): %s' % (rc, mask(cmd)))

os.chdir(WORKDIR)
FIRST_TIME = not os.path.exists('data/evaluation.csv')   # 처음이면 새로 나누고, 이미 있으면 기존 분할을 지킨다
cmd = sys.executable + ' -m kore split --questions data/evaluation_raw.csv --knowledge data/knowledge.csv --out data/evaluation.csv --seed 0'
if not FIRST_TIME:
    cmd += ' --keep-from data/evaluation.csv'
sh(cmd, stream=True)
sh(sys.executable + ' -m kore validate --knowledge data/knowledge.csv --questions data/evaluation.csv', stream=True)
sh(sys.executable + ' -m kore stats --knowledge data/knowledge.csv --questions data/evaluation.csv --stage week1', stream=True)
print('[실패] (필수)가 남아 있으면 고쳐서 다시 올린다. 통과하면 4번 셀로 evaluation.csv를 내려받는다.')

## 4. 내려받기

In [ ]:
# 4. 결과 파일 내려받기 → GitHub 웹에서 data/ 폴더에 올린다(START.md 6절)
from google.colab import files
WHO = 'B'   # B면 knowledge.csv와 source_log.csv, C면 evaluation.csv와 evaluation_raw.csv
names = ['knowledge.csv', 'source_log.csv'] if WHO == 'B' else ['evaluation.csv', 'evaluation_raw.csv']
for n in names:
    files.download('/content/kore/data/' + n)